## AdaBoost

In [2]:
import numpy as np

## 算法类

In [ ]:
class DecisionStump:
    """
    决策树桩：只根据一个特征和一个阈值进行分类
    形式大致是：
        如果 x_j < threshold:
            预测为 -1
        否则:
            预测为 +1
    或者反过来。
    """

    def __init__(self, feature_index=None, threshold=None, polarity=1):
        self.feature_index = feature_index
        self.threshold = threshold
        self.polarity = polarity

    def predict(self, X):
        """
        根据当前树桩进行预测
        参数
        ----------
        X : ndarray, shape = (n_samples, n_features)
        返回
        ----------
        y_pred : ndarray, shape = (n_samples,)
            预测结果，只能是 -1 或 +1
        """

        X = np.asarray(X)
        n_samples = X.shape[0]
        y_pred = np.ones(n_samples)
        feature_values = X[:, self.feature_index]
        if self.polarity == 1:
            y_pred[feature_values < self.threshold] = -1
        else:
            y_pred[feature_values < self.threshold] = 1
            y_pred[feature_values >= self.threshold] = -1

        return y_pred

In [ ]:
class SimpleAdaBoostClassifier:
    """
    简单版 AdaBoost 二分类器
    只用于学习 AdaBoost 算法原理。
    目前支持：
    - 二分类问题
    - 数值型特征
    - 标签可以是 {-1, +1}
    """
    def __init__(self, n_estimators=10):
        """
        参数
        ----------
        n_estimators : int
            弱分类器的个数，也就是 AdaBoost 迭代轮数 M
        """
        self.n_estimators = n_estimators
        self.stumps = [] # 创建一个空列表，用来保存每一轮训练出来的弱分类器
        self.alphas = [] # 创建一个空列表，用来保存每个弱分类器的权重

    def _build_stump(self, X, y, sample_weight):
        """
        训练一个最优决策树桩
        这一步对应 AdaBoost 算法中的：
            使用当前权值分布 D_m 的训练数据集，
            学习得到基本分类器 G_m(x)
        目标是找到一个加权分类误差率最小的树桩。
        """
        n_samples, n_features = X.shape
        best_stump = None
        best_error = float("inf")
        best_pred = None

        for feature_index in range(n_features):

            feature_values = X[:, feature_index]
            thresholds = np.unique(feature_values)

            for threshold in thresholds:

                for polarity in [1, -1]:

                    stump = DecisionStump(
                        feature_index=feature_index,
                        threshold=threshold,
                        polarity=polarity
                    )

                    y_pred = stump.predict(X)

                    error = np.sum(sample_weight[y_pred != y])

                    if error < best_error:
                        best_error = error
                        best_stump = stump
                        best_pred = y_pred

        return best_stump, best_error, best_pred

    def fit(self, X, y):
        """
        训练 AdaBoost 模型
        参数
        ----------
        X : ndarray, shape = (n_samples, n_features)
            训练数据
        y : ndarray, shape = (n_samples,)
            标签，必须是 -1 或 +1
        返回
        ----------
        self
        """
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)
        n_samples, n_features = X.shape
        if not set(np.unique(y)).issubset({-1, 1}):
            raise ValueError("这个简单版本要求 y 的标签必须是 -1 和 +1")
        # ===============================
        # 第 1 步：初始化样本权值分布 D_1
        # ===============================
        sample_weight = np.ones(n_samples) / n_samples
        self.stumps = []
        self.alphas = []
        # ===============================
        # 第 2 步：迭代训练 M 个基本分类器
        # ===============================
        for m in range(self.n_estimators):
            # -------------------------------
            # 第 2(a) 步：训练基本分类器 G_m(x)
            # -------------------------------
            stump, error, y_pred = self._build_stump(
                X,
                y,
                sample_weight
            )
            # 为了避免除以 0
            error = max(error, 1e-10)
            # 如果弱分类器还不如随机猜测，就停止
            if error >= 0.5:
                break
            # -------------------------------
            # 第 2(c) 步：计算基本分类器系数 alpha_m
            # -------------------------------
            alpha = 0.5 * np.log((1 - error) / error)
            # -------------------------------
            # 第 2(d) 步：更新训练样本权值分布
            # -------------------------------
            sample_weight = sample_weight * np.exp(
                -alpha * y * y_pred)
            # 规范化，使权值和为 1
            sample_weight = sample_weight / np.sum(sample_weight)
            # 保存当前弱分类器和它的权重
            self.stumps.append(stump)
            self.alphas.append(alpha)
            print(f"第 {m + 1} 轮")
            print(f"加权误差率 error = {error:.4f}")
            print(f"分类器权重 alpha = {alpha:.4f}")
            print(f"样本权重 = {sample_weight}")
            print("-" * 40)
        return self

    def decision_function(self, X):
        """
        计算加法模型：
            f(x) = alpha_1 G_1(x) + ... + alpha_M G_M(x)
        返回的是未取 sign 之前的结果。
        """
        X = np.asarray(X, dtype=float)
        f = np.zeros(X.shape[0])
        for alpha, stump in zip(self.alphas, self.stumps):
            f += alpha * stump.predict(X)
        return f

    def predict(self, X):
        """
        最终分类器：
            G(x) = sign(f(x))
        """
        f = self.decision_function(X)
        return np.sign(f)

## 测试

In [ ]:
if __name__ == "__main__":

    X = np.array([[0, 1],[1, 2],[2, 1],[3, 3],[4, 3],[5, 2]])
    y = np.array([-1,-1,-1,1,1,1])
    model = SimpleAdaBoostClassifier(n_estimators=5)
    model.fit(X, y)
    y_pred = model.predict(X)
    print("最终预测结果：")
    print(y_pred)
    print("真实标签：")
    print(y)
    print("加法模型 f(x)：")
    print(model.decision_function(X))

### sklearn

In [6]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. 加载数据
iris = load_iris()
X = iris.data
y = iris.target

# 2. 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# 3. 建立模型
clf = AdaBoostClassifier(
    n_estimators=100,
    learning_rate=0.5,
    random_state=42
)

# 4. 训练
clf.fit(X_train, y_train)

# 5. 预测
y_pred = clf.predict(X_test)

# 6. 评估
print("准确率:", accuracy_score(y_test, y_pred))
print("混淆矩阵:")
print(confusion_matrix(y_test, y_pred))
print("分类报告:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

准确率: 0.9666666666666667
混淆矩阵:
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]
分类报告:
              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      0.90      0.95        10
   virginica       0.91      1.00      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



In [8]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# 1. 数据
data = fetch_california_housing()
X = data.data
y = data.target

# 2. 划分
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# 3. 基学习器
base_tree = DecisionTreeRegressor(
    max_depth=4,
    random_state=42
)

# 4. AdaBoost 回归
reg = AdaBoostRegressor(
    estimator=base_tree,
    n_estimators=100,
    learning_rate=0.5,
    loss="linear",
    random_state=42
)

# 5. 训练
reg.fit(X_train, y_train)

# 6. 预测
y_pred = reg.predict(X_test)

# 7. 评估
print("MSE:", mean_squared_error(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MSE: 0.7367293548685884
MAE: 0.7483253382392666
R2: 0.4377867008683213
